# OpenPlaque — Secondary Branch Anchored Lumen Validation v3

This experiment addresses the v2 vesselness false-positive risk.

It **requires the 3-D graph to pass through the accepted secondary-branch endpoint (~13.84 mm)** and then validates every distal candidate with serial source-CCTA orthogonal planes. A vesselness corridor is not accepted unless it remains a compact, centered, coronary-like contrast lumen.

Research use only. Vessel identity is not assigned automatically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse controls immediately after Drive mount.
REUSE_INPUTS = True
REUSE_ANCHORED_SEARCH = False
REUSE_FIGURES = False
REUSE_REPORT = False


In [ ]:
!pip -q install scipy pandas matplotlib


In [ ]:
import os, shutil
if os.path.exists('/content/OpenPlaque'):
    shutil.rmtree('/content/OpenPlaque')
!git clone -q --depth 1 --branch secondary-anchored-lumen-validation-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%cd /content/OpenPlaque


In [ ]:
import sys
sys.path.insert(0, '/content/OpenPlaque/src')
from openplaque.secondary_anchored_lumen_validation import (
    SecondaryAnchoredLumenValidationWorkflow,
    synthetic_lumen_validator_self_test,
)
self_test = synthetic_lumen_validator_self_test()
print('Synthetic compact-lumen validator self-test:', self_test)
assert self_test['passed'], 'Synthetic lumen validator self-test failed'


In [ ]:
reuse = {
    'inputs': REUSE_INPUTS,
    'anchored_search': REUSE_ANCHORED_SEARCH,
    'figures': REUSE_FIGURES,
    'report': REUSE_REPORT,
}
wf = SecondaryAnchoredLumenValidationWorkflow(reuse=reuse)
display(wf.cache_status())


In [ ]:
inputs = wf.load_inputs(control_arc_mm=12.7)
print('Mandatory accepted endpoint:', inputs['mandatory_anchor_accepted_endpoint_zyx'])
print('Anchor radius (mm):', inputs['mandatory_anchor_radius_mm'])
print('Lumen calibration:')
display(wf.calibration)


In [ ]:
summary = wf.run(max_cost=150.0, min_projection_mm=1.5)
display(summary)
if wf.candidates is not None:
    display(wf.candidates.head(12))
if wf.best_qc is not None and len(wf.best_qc):
    display(wf.best_qc)


In [ ]:
figs = wf.make_figures()
print('Figures:', figs)
from IPython.display import Image, display
for name in figs:
    display(Image(filename=str(wf.out / name)))


In [ ]:
report = wf.make_report()
zip_path = wf.package()
print('Report:', report)
print('ZIP:', zip_path)


In [ ]:
from IPython.display import HTML, display
from urllib.parse import quote
final_name = 'OPENPLAQUE_SECONDARY_ANCHORED_LUMEN_VALIDATION_REPORT_BACK.zip'
search_url = 'https://drive.google.com/drive/u/0/search?q=' + quote(final_name)
colab_url = 'https://colab.research.google.com/github/pazzani/OpenPlaque/blob/secondary-anchored-lumen-validation-from-main/colab/OpenPlaque_Secondary_Anchored_Lumen_Validation.ipynb'
print('Colab:', colab_url)
print('Drive search:', search_url)
display(HTML(f'<p><a href="{search_url}" target="_blank">Open final result in Google Drive search</a></p>'))
